# Lesson 4: Decision Trees

Non-linear, interpretable classification with recursive partitioning.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Gini Impurity and Entropy

In [ ]:
def gini(p):
    return 1 - p**2 - (1-p)**2

def entropy(p):
    return -p * np.log2(p) - (1-p) * np.log2(1-p) if 0 < p < 1 else 0

p = np.linspace(0.001, 0.999, 100)
plt.figure(figsize=(8, 4))
plt.plot(p, [gini(pi) for pi in p], label='Gini')
plt.plot(p, [entropy(pi) for pi in p], label='Entropy')
plt.xlabel('p (proportion of class 1)')
plt.ylabel('Impurity')
plt.title('Gini vs. Entropy')
plt.legend()
plt.show()

## 2. Visualizing a Decision Tree on Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X, y)

plt.figure(figsize=(16, 8))
plot_tree(tree, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, rounded=True)
plt.show()

print(export_text(tree, feature_names=iris.feature_names))

## 3. Effect of Tree Depth on Overfitting

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

depths = [1, 2, 3, 4, 5, 10, 20, None]
train_scores, test_scores = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, t.predict(X_train)))
    test_scores.append(accuracy_score(y_test, t.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(range(len(depths)), train_scores, 'o-', label='Train')
plt.plot(range(len(depths)), test_scores, 's-', label='Test')
plt.xticks(range(len(depths)), [str(d) for d in depths])
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Depth vs. Overfitting')
plt.legend()
plt.grid(True)
plt.show()

## 4. Feature Importance

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=4, min_samples_split=10, random_state=42)
best_tree.fit(X_train, y_train)

importance = pd.DataFrame({
    'feature': data.feature_names,
    'importance': best_tree.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

print(f"Test accuracy: {accuracy_score(y_test, best_tree.predict(X_test)):.3f}")

## 5. Decision Boundary Comparison

In [ ]:
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                            n_clusters_per_class=1, class_sep=0.8, random_state=42)

models = {
    'Depth 1': DecisionTreeClassifier(max_depth=1),
    'Depth 3': DecisionTreeClassifier(max_depth=3),
    'Depth 10': DecisionTreeClassifier(max_depth=10),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 100),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 100))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X, y)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', alpha=0.7)
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 6. Exercises

### Level 1
Explain: What is Gini impurity and what does Gini=0 mean?

### Level 2
Train decision trees with max_depth 1-10 on breast cancer. Plot train vs test accuracy. What's the optimal depth?

### Level 3
Your tree achieves 100% train accuracy but 62% test accuracy. What strategies fix this?

In [ ]:
# Your Level 2 code

## 7. Coding Challenge

Write `tree_depth_tuner(X_train, X_val, y_train, y_val, max_depths)` that finds the optimal depth.

In [ ]:
def tree_depth_tuner(X_train, X_val, y_train, y_val, max_depths):
    best_acc, best_d, best_tree = 0, None, None
    for d in max_depths:
        t = DecisionTreeClassifier(max_depth=d, random_state=42)
        t.fit(X_train, y_train)
        acc = accuracy_score(y_val, t.predict(X_val))
        print(f"Depth {d}: Val accuracy = {acc:.3f}")
        if acc > best_acc:
            best_acc, best_d, best_tree = acc, d, t
    print(f"Best depth: {best_d}, accuracy: {best_acc:.3f}")
    return best_d, best_tree

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)
best_d, best_tree = tree_depth_tuner(X_tr, X_val, y_tr, y_val, range(1, 15))

## Summary

- Decision trees split data using recursive partitioning
- Gini and entropy measure node purity
- Information gain guides split selection
- Deeper trees → more overfitting
- Pruning (max_depth, min_samples_leaf) prevents overfitting
- Trees are interpretable but unstable
- Feature importance identifies key predictors